# FXS / 12 — Fading moves with OCO limit orders

Pre-registration: `discussions/FX_Intraday_Seasonality/007_claude_fxs12_range_fade_preregistration.md`.

At the top of each New York hour h:
- place a **sell limit** at mid + D and a **buy limit** at mid − D (OCO; the first fill cancels the other);
- take profit back at the starting mid (+D);
- stop S beyond the fill;
- close everything at h + 2 h.

**Grid:** D ∈ {5, 10, 15, 20} pips, S ∈ {D, 2D}, 24 hours, EURUSD / USDJPY / EURJPY: 576 tests.

**Windows:** discovery 2020–2023 → confirmation 2024–2025 → 2026 one look.

Costs and fills: fills on the bid/ask touch at the level (strict variant: traded through by 0.2 pip), spread × 0.9, $3 per lot.

**What would show an edge:** among trades that reach the take-profit or the stop, a take-profit share clearly above the random-walk rate (S − spread)/(D + S), and positive net pips that survive confirmation. On a random walk this trade loses roughly the costs (more with the conservative same-minute rules: on synthetic random-walk data the take-profit share sits a few points below the benchmark).

In [ ]:
import hashlib, json, os, sys, tempfile
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

PAIRS = ["EURUSD", "USDJPY", "EURJPY"]
DS = [5, 10, 15, 20]
SMULT = [1, 2]
HOURS = list(range(24))
HOLD = 120                       # minutes from placement to the time exit
COMMISSION_PER_LOT = 3.0
SPREAD_FACTOR = 0.9
STRICT_PIPS = 0.2
Q = 0.05
MAX_CAND = 10
DISC_YEARS, CONF_YEARS = (2020, 2023), (2024, 2025)
LOAD = ((dt.date(2020, 1, 1), dt.date(2020, 12, 31), "discovery"), (dt.date(2021, 1, 1), dt.date(2025, 12, 31), "confirmation"))
HOLDOUT = (dt.date(2026, 1, 1), dt.date(2026, 11, 6), "holdout")

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs12_")
    os.environ.pop("DUKASCOPY_DIR", None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg = F.FXSConfig()
OUT = Path.cwd() / ("12_synthetic" if SYNTHETIC else "12"); OUT.mkdir(exist_ok=True)
CONFIGS = [(D, D * k) for D in DS for k in SMULT]
pd.set_option("display.width", 240, "display.max_columns", 40, "display.max_rows", 200)
np.seterr(invalid="ignore", divide="ignore")

## 0. Synthetic inputs (smoke run only)

A pure random walk: the fade must lose about its costs, and the take-profit rate must sit near the random-walk benchmark.

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(PAIRS):
        F.write_synthetic_candles(base, p, dt.date(2019, 12, 28), dt.date(2026, 11, 6), seed=91 + i, price=150.0 if p.endswith("JPY") else 1.10)

## 1. The simulator

All days of one (pair, hour) are simulated at once. Matrices are days × 120 minutes after placement.

In [ ]:
def windows(m, hour, dates):
    """Minute matrices for placements at New York hour `hour` on each trading-day label."""
    t0, _ = F.entry_exit_times("NY", 60 * hour, 60 * hour + HOLD, dates)
    idx = m.index
    i0 = idx.searchsorted(t0)
    ok = (~pd.isna(t0)) & (i0 > 0) & (i0 + HOLD < len(idx))
    ok &= np.asarray(idx[np.clip(i0, 0, len(idx) - 1)] == t0)
    ny = pd.DatetimeIndex(t0).tz_convert(F.NY)
    crosses = (ny.weekday == 4) & (ny.hour >= 15)                     # 2 h window reaches the Friday 17:00 close
    ok &= ~np.asarray(crosses)
    i0 = i0[ok]
    rows = i0[:, None] + np.arange(HOLD)[None, :]
    X = {c: m[c].to_numpy()[rows] for c in F.OHLC_COLS}
    hs = (X["ask_c"] - X["bid_c"]) / 2 * (1 - SPREAD_FACTOR)            # tighten every quote around the mid
    for c in F.OHLC_COLS:
        X[c] = X[c] + hs if c.startswith("bid") else X[c] - hs
    b0, a0 = m["bid_c"].to_numpy()[i0 - 1], m["ask_c"].to_numpy()[i0 - 1]
    X["M0"] = 0.5 * (b0 + a0)
    X["spread0"] = (a0 - b0) * SPREAD_FACTOR
    X["dates"] = pd.DatetimeIndex(pd.to_datetime(dates))[ok]
    return X

def first(mask):
    K = mask.shape[1]
    return np.where(mask.any(1), mask.argmax(1), K)

def simulate(X, D, S, pip, comm_pips, through=0.0):
    """Returns DataFrame (one row per placement): filled, side (+1 long / -1 short), net pips, reason."""
    n, K = X["bid_h"].shape
    k = np.arange(K)[None, :]
    M0 = X["M0"][:, None]
    U, L, th = M0 + D * pip, M0 - D * pip, through * pip
    fs = first(X["bid_h"] >= U + th)
    fb = first(X["ask_l"] <= L - th)
    same = (fs == fb) & (fs < K)
    close_up = X["ask_c"][np.arange(n), np.minimum(fs, K - 1)] + X["bid_c"][np.arange(n), np.minimum(fs, K - 1)] >= 2 * X["M0"]
    short = (fs < fb) | (same & close_up)
    f = np.where(short, fs, fb)
    filled = f < K
    fcol = f[:, None]
    # short leg
    s_stop = U + S * pip
    ks_s = first((X["ask_h"] >= s_stop) & (k >= fcol))
    kt_s = first((X["ask_l"] <= M0) & (k > fcol))
    # long leg
    l_stop = L - S * pip
    ks_l = first((X["bid_l"] <= l_stop) & (k >= fcol))
    kt_l = first((X["bid_h"] >= M0) & (k > fcol))
    ks, kt = np.where(short, ks_s, ks_l), np.where(short, kt_s, kt_l)
    stop_px = np.where(short, s_stop[:, 0], l_stop[:, 0])
    rr = np.arange(n)
    ksc = np.minimum(ks, K - 1)
    gap_open = np.where(short, X["ask_o"][rr, ksc], X["bid_o"][rr, ksc])
    stop_fill = np.where(ks > f, np.where(short, np.maximum(stop_px, gap_open), np.minimum(stop_px, gap_open)), stop_px)
    time_px = np.where(short, X["ask_c"][:, -1], X["bid_c"][:, -1])
    is_stop = filled & (ks < K) & (ks <= kt)
    is_tp = filled & ~is_stop & (kt < K)
    exit_px = np.where(is_stop, stop_fill, np.where(is_tp, X["M0"], time_px))
    entry_px = np.where(short, U[:, 0], L[:, 0])
    gross = np.where(short, entry_px - exit_px, exit_px - entry_px) / pip
    net = np.where(filled, gross - comm_pips, np.nan)
    reason = np.where(~filled, "none", np.where(is_stop, "stop", np.where(is_tp, "tp", "time")))
    return pd.DataFrame({"date": X["dates"], "filled": filled, "side": np.where(short, -1, 1), "net": net, "reason": reason,
                         "spread0": X["spread0"] / pip})

def stats_of(df, years=None):
    d = df[df["filled"]]
    if years is not None:
        d = d[(d["date"].dt.year >= years[0]) & (d["date"].dt.year <= years[1])]
    x = d["net"].to_numpy()
    if len(x) < 10:
        return {"fills": len(x)}
    m, se, t = F.nw_tstats(x[:, None], 5)
    return {"fills": len(x), "mean net": m[0], "t": t[0], "p": 2 * stats.norm.sf(abs(t[0])), "tp rate": (d["reason"] == "tp").mean(),
            "stop rate": (d["reason"] == "stop").mean(), "time rate": (d["reason"] == "time").mean(), "median spread": d["spread0"].median(),
            "years same sign": int(sum(np.sign(g["net"].mean()) == np.sign(m[0]) for _, g in d.groupby(d["date"].dt.year)))}

## 2. Discovery 2020–2023 (and confirmation numbers computed in the same pass, shown only in §3)

In [ ]:
RES, TRADES = [], {}
for p in PAIRS:
    m = pd.concat([F.load_minutes_ohlc(p, a, b, st, cfg.min_file_share) for a, b, st in LOAD])
    m = m[~m.index.duplicated(keep="last")].sort_index()
    pip = F.pip_size(p)
    dates = pd.bdate_range(LOAD[0][0], LOAD[-1][1])
    price = float(0.5 * (m["bid_c"].iloc[-1] + m["ask_c"].iloc[-1]))
    comm = F.bps_to_pips(F.commission_bps_per_lot(p, COMMISSION_PER_LOT), price, p)
    for h in HOURS:
        X = windows(m, h, dates)
        for D, S in CONFIGS:
            df = simulate(X, D, S, pip, comm)
            TRADES[(p, h, D, S)] = df
            r = {"pair": p, "hour": h, "time": F.slot_label("NY", 60 * h), "D": D, "S": S}
            r.update({f"{k}": v for k, v in stats_of(df, DISC_YEARS).items()})
            r["rw tp rate"] = (S - r.get("median spread", np.nan)) / (D + S)
            RES.append(r)
    del m
    print(p, "done")
R = pd.DataFrame(RES)
ok = R["p"].notna()
R["q"] = np.nan
# one-sided (mean > 0): with costs every configuration has a negative expectation under the null, so a two-sided
# family is dominated by significant losers and BH then passes weak positives (FXS 12 first run; amendment in 007)
R["p"] = stats.norm.sf(R["t"])
R.loc[ok, "q"] = F.bh_reject(R.loc[ok, "p"].to_numpy(), Q)[1]
R["tp share of resolved"] = R["tp rate"] / (R["tp rate"] + R["stop rate"])        # time exits left out: the benchmark has no clock
R["tp excess"] = R["tp share of resolved"] - R["rw tp rate"]
R.to_csv(OUT / "12_discovery.csv", index=False)
print("mean net pips per filled trade, all 576 tests:", R["mean net"].describe().round(2).to_dict())
print("take-profit share of resolved trades minus random-walk rate:", R["tp excess"].describe().round(3).to_dict())

### Heatmaps: mean net pips per filled trade, 2020–2023 (hour × configuration)

In [ ]:
fig, axes = plt.subplots(1, len(PAIRS), figsize=(20, 7), sharey=True)
for ax, p in zip(axes, PAIRS):
    h = R[R["pair"] == p].pivot(index="hour", columns=["D", "S"], values="mean net")
    lim = np.nanmax(np.abs(h.to_numpy()))
    im = ax.imshow(h.to_numpy(), aspect="auto", cmap="RdBu", vmin=-lim, vmax=lim)
    ax.set_xticks(range(h.shape[1])); ax.set_xticklabels([f"D{d}/S{s}" for d, s in h.columns], rotation=45, fontsize=8)
    ax.set_yticks(range(24)); ax.set_yticklabels([F.slot_label("NY", 60 * k).split()[1] for k in range(24)], fontsize=8)
    ax.set_title(f"{p} (NY placement time)"); plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout(); plt.savefig(OUT / "12_heatmap_discovery.png", dpi=110); plt.show()
display(R.sort_values("t", ascending=False).head(20).round(3))

## 3. Candidates and confirmation 2024–2025

In [ ]:
surv = R[(R["q"] <= Q) & (R["mean net"] > 0) & (R["years same sign"] >= 3)].sort_values("t", ascending=False)
cand = surv.drop_duplicates(["pair", "hour"]).head(MAX_CAND).copy()
print(f"{len(surv)} configurations pass discovery; {len(cand)} candidates")
rows = []
for _, c in cand.iterrows():
    key = (c["pair"], c["hour"], c["D"], c["S"])
    st = stats_of(TRADES[key], CONF_YEARS)
    rows.append({"pair": c["pair"], "time": c["time"], "D": c["D"], "S": c["S"], "disc mean": c["mean net"], "disc t": c["t"],
                 "conf fills": st.get("fills"), "conf mean": st.get("mean net"), "conf t": st.get("t"), "conf tp rate": st.get("tp rate"),
                 "conf stop rate": st.get("stop rate"), "rw tp rate": c["rw tp rate"]})
CF = pd.DataFrame(rows)
if len(CF):
    CF["p one-sided"] = stats.norm.sf(CF["conf t"])
    CF["q"] = F.bh_reject(CF["p one-sided"].to_numpy(), Q)[1]
    CF["CONFIRMED"] = (CF["conf mean"] > 0) & (CF["q"] <= Q)
    # strict fills (level traded through by 0.2 pip), confirmation years
    strict = []
    for _, c in CF.iterrows():
        m = pd.concat([F.load_minutes_ohlc(c["pair"], a, b, st_, cfg.min_file_share) for a, b, st_ in LOAD[1:]])
        X = windows(m, int(cand[(cand["pair"] == c["pair"]) & (cand["time"] == c["time"])]["hour"].iloc[0]), pd.bdate_range("2024-01-01", "2025-12-31"))
        price = float(X["M0"][-1]); pip = F.pip_size(c["pair"])
        df = simulate(X, c["D"], c["S"], pip, F.bps_to_pips(F.commission_bps_per_lot(c["pair"], COMMISSION_PER_LOT), price, c["pair"]), STRICT_PIPS)
        strict.append(stats_of(df, CONF_YEARS).get("mean net"))
        del m
    CF["conf mean (strict fills)"] = strict
display(CF.round(3) if len(CF) else "no candidates"); CF.to_csv(OUT / "12_confirmation.csv", index=False)

## 4. Risk of the candidates (2020–2025)

Worst trade, worst day (all candidate hours of the pair summed), longest run of losing trades, yearly mean net pips.

In [ ]:
risk, yearly = [], []
for _, c in cand.iterrows():
    df = TRADES[(c["pair"], c["hour"], c["D"], c["S"])]
    d = df[df["filled"]]
    lose = (d["net"] < 0).astype(int).to_numpy()
    run = best = 0
    for v in lose:
        run = run + 1 if v else 0; best = max(best, run)
    risk.append({"pair": c["pair"], "time": c["time"], "D": c["D"], "S": c["S"], "fills/yr": len(d) / 6, "worst trade": d["net"].min(),
                 "p5 trade": d["net"].quantile(0.05), "longest losing run": best, "stop rate": (d["reason"] == "stop").mean()})
    yearly.append(d.groupby(d["date"].dt.year)["net"].mean().rename(f"{c['pair']} {c['time']} D{c['D']}/S{c['S']}"))
if risk:
    display(pd.DataFrame(risk).round(2)); display(pd.DataFrame(yearly).round(2))
    pd.DataFrame(risk).to_csv(OUT / "12_risk.csv", index=False)

## 5. 2026, one look (confirmed configurations only)

In [ ]:
conf = CF[CF["CONFIRMED"]] if len(CF) else pd.DataFrame()
ff = OUT / "12_confirmed_frozen.csv"
(conf if len(conf) else pd.DataFrame(columns=["pair", "time", "D", "S"]))[["pair", "time", "D", "S"]].to_csv(ff, index=False, lineterminator="\n")
SHA = hashlib.sha256(ff.read_bytes()).hexdigest()
print("frozen", ff.name, SHA)
H = []
for _, c in conf.iterrows():
    hour = int(cand[(cand["pair"] == c["pair"]) & (cand["time"] == c["time"])]["hour"].iloc[0])
    m = F.load_minutes_ohlc(c["pair"], *HOLDOUT, cfg.min_file_share)
    X = windows(m, hour, pd.bdate_range(*HOLDOUT[:2]))
    pip = F.pip_size(c["pair"])
    comm = F.bps_to_pips(F.commission_bps_per_lot(c["pair"], COMMISSION_PER_LOT), float(X["M0"][-1]), c["pair"])
    st = stats_of(simulate(X, c["D"], c["S"], pip, comm))
    H.append({"pair": c["pair"], "time": c["time"], "D": c["D"], "S": c["S"], **st, "holds": st.get("mean net", -1) > 0})
H = pd.DataFrame(H)
display(H.round(3) if len(H) else "nothing confirmed, so 2026 is not read"); H.to_csv(OUT / "12_holdout_2026.csv", index=False)

In [ ]:
summary = {"study": "FXS12", "prereg": "discussions/FX_Intraday_Seasonality/007_claude_fxs12_range_fade_preregistration.md", "synthetic": SYNTHETIC,
           "n_tests": int(len(R)), "discovery_pass": int(len(surv)), "candidates": cand.round(4).to_dict(orient="records"),
           "confirmation": CF.round(4).to_dict(orient="records") if len(CF) else [], "frozen_sha256": SHA,
           "holdout_2026": H.round(4).to_dict(orient="records") if len(H) else [],
           "all_tests_mean_net": R["mean net"].describe().round(4).to_dict(), "tp_excess": R["tp excess"].describe().round(4).to_dict()}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("written", OUT / "summary.json")